# Аудио-пайплайн MVP
Транскрипция · Техническое качество · Флаг Унылость (в разработке)

In [ ]:
%cd "/content"

In [ ]:
import os

!pip install -U "yt-dlp[default]" -q

os.environ["PATH"] += ":/root/.deno/bin"
!curl -fsSL https://deno.land/install.sh | sh

!yt-dlp --version

In [ ]:
!yt-dlp --dump-json "https://www.youtube.com/watch?v=qf-tkyydSCc"

In [ ]:
from google.colab import userdata

REPO_URL = userdata.get('REPO_URL')
token = userdata.get("GITHUB_TOKEN")

!git clone https://{token}@{REPO_URL}

# Определяем название папки из URL
repo_name = REPO_URL.split("/")[-1].replace(".git", "")
%cd {repo_name}"/pipeline"

In [ ]:
!pip install -r requirements.txt -q
!pip install faster-whisper --no-deps

# silero-VAD устанавливается через torch.hub автоматически при первом вызове
# но можно прогреть кэш заранее
import torch
torch.hub.load(
    repo_or_dir='snakers4/silero-vad',
    model='silero_vad',
    force_reload=False,
    trust_repo=True,
)
print("Зависимости установлены")

In [ ]:
import subprocess, os, sys

subprocess.run(["rm", "-rf", "/tmp/DOVER"], check=True)
subprocess.run(["git", "clone", "https://github.com/QualityAssessment/DOVER.git", "/tmp/DOVER", "-q"], check=True)

# Не переустанавливаем torch в Colab
subprocess.run(["sed", "-i", "/^torch/d", "/tmp/DOVER/requirements.txt"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-r", "/tmp/DOVER/requirements.txt", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "/tmp/DOVER", "--no-deps", "-q"], check=True)

os.makedirs("/tmp/DOVER/pretrained_weights", exist_ok=True)
subprocess.run([
    "wget", "-q",
    "https://github.com/QualityAssessment/DOVER/releases/download/v0.1.0/DOVER.pth",
    "-O", "/tmp/DOVER/pretrained_weights/DOVER.pth"
], check=True)

print("Установка готова ✅")

In [ ]:
import sys
sys.path.insert(0, "/tmp/DOVER")

from dover.models import DOVER

print("DOVER импортируется ✅")

In [ ]:
result = subprocess.run(
    ["python", "evaluate_one_video.py", "-v", "/tmp/DOVER/demo/e4dv6ZsFzHE_000243_000253.mp4",
     "-f"
     ],
    cwd="/tmp/DOVER",
    capture_output=True,
    text=True,
)

print(result.stdout)

In [ ]:
import torch
from config import DEVICE, COMPUTE_TYPE

print(f"CUDA доступна: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Пайплайн использует: device={DEVICE}, compute_type={COMPUTE_TYPE}")

## 5. Запуск пайплайна

In [ ]:
import json
import sys
sys.path.insert(0, 'pipeline')  # добавляем папку pipeline в path

from pipeline import run

# URL = "https://www.youtube.com/watch?v=UzfAt6DoN3E" # новее видео трушина, звук значительно лучше, но не оч громкий (result)
URL = "https://www.youtube.com/watch?v=WTjfi-eqL7E" # запись со стрима очень давнишняя, должен быть хуже скор (result_bad)
# URL = "https://youtu.be/uuesXwc1DJY?si=BkWS56xBDh9iIM2X" # мфти прям с ужасным качеством звука, по идее тут и клипинг должен сработать (result_mipt_bad)
# URL = "https://www.youtube.com/watch?v=SEQ3vqeO30k" # ноунейм, звук вообще ужас, там голоса не слышно по сравнению с физтехом (мел и шумы, у физтеха такого не было) (result_noname_bad)

# skip_video=True — только аудио-модули, быстрее для отладки
result = run(URL, skip_video=True)
#print(json.dumps(result, ensure_ascii=False, indent=2))

## 6. Просмотр отдельных метрик

In [ ]:
# Транскрипция
print(f"Язык: {result['transcription']['language']}")
print(f"WPM: {result['transcription']['wpm']}")
print(f"Первые 300 символов: {result['transcription']['text'][:300]}")

In [ ]:
# Качество звука
aq = result['audio_quality']
print(f"DNSMOS ovrl: {aq['dnsmos']['ovrl_mos']} ({aq['dnsmos']['quality']})")
print(f"DNSMOS sig:  {aq['dnsmos']['sig_mos']}")
print(f"DNSMOS bak:  {aq['dnsmos']['bak_mos']}")
print(f"LUFS:        {aq['lufs']['value']} dB ({aq['lufs']['quality']})")
print(f"Клиппинг:    {aq['clipping']['level']} (ratio={aq['clipping']['ratio']}, crest={aq['clipping']['crest_factor']}, flat={aq['clipping']['flattened_peaks_ratio']})")
print(f"SNR:         {aq['snr']['db']} dB ({aq['snr']['quality']})")

In [ ]:
dl = result['dullness']
print(f"Унылость:    flag={dl['flag']}  score={dl['score']}")
print(f"  acoustic:  {dl['acoustic_score']}")
print(f"  linguistic:{dl['linguistic_score']}")
print(f"  WPM:       {dl['components']['wpm']}")
print(f"  F0 std:    {dl['components']['f0_std']}")
print(f"  HNR log:   {dl['components']['hnr_log']}")
print(f"  TTR global:{dl['components']['ttr_global']}")
print(f"  fillers:   {dl['components']['filler_ratio']}")

In [ ]:
# Качество видео
vq = result['video_quality']

if vq.get('skipped'):
    print("Модуль качества видео был пропущен (skip_video=True)")
else:
    print(f"Разрешение:  {vq['resolution']['width']}x{vq['resolution']['height']} ({vq['resolution']['level']})")
    print(f"FPS:         {vq['resolution']['fps']}")
    print(f"Кодек:       {vq['resolution']['codec']}")
    print()
    print(f"DOVER:       score={vq['dover']['score']} ({vq['dover']['quality']})")
    print(f"BRISQUE:     median={vq['brisque']['median']} p90={vq['brisque']['p90']} ({vq['brisque']['level']})")
    print(f"Blur:        median={vq['blur']['median']} p10={vq['blur']['p10']} ({vq['blur']['level']})")
    print()
    print(f"Экспозиция:  brightness={vq['exposure']['mean_brightness']} ({vq['exposure']['level']})")
    print(f"Контраст:    {vq['contrast']['median']} ({vq['contrast']['level']})")
    print(f"Мерцание:    {vq['flicker']['mean']} ({vq['flicker']['level']})")
    print(f"Стабильность:{vq['stability']['motion_mean']} ({vq['stability']['level']})")
    print(f"Консистент.: {vq['consistency']['level']}")
    print()
    print(f"Спикер:      presence={vq['speaker']['face_presence_ratio']} size={vq['speaker']['face_size_median']}")
    print(f"Доска:       detected={vq['board']['detected']} readability={vq['board']['readability_level']} glare={vq['board']['glare_level']}")

In [ ]:
with open('result.json', 'w', encoding='utf-8') as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print("Сохранено в result.json")